In [5]:
from mecon.data.data_management import CachedFileDataManager
from mecon.etl.dataset import Dataset
import pathlib
from mecon.app.current_data import WorkingDataManager
datasets_dir = pathlib.Path("/Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets")#config.DEFAULT_DATASETS_DIR_PATH
dataset = Dataset.from_dirpath(datasets_dir / 'shared')
data_manager = CachedFileDataManager(dataset)
dataset

INFO:root:MECON_ROOT_DIRPATH not found in environment variables. Using default value relative to __file__='/Users/wimpole/PycharmProjects/mecon/mecon/config.py'.
INFO:root:MECON_ROOT_DIRPATH set to: MECON_ROOT_DIRPATH=PosixPath('/Users/wimpole/PycharmProjects/mecon')


DatasetV2(shared): /Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/shared

In [6]:
transactions = data_manager.get_transactions()
transactions.size()


7643

In [8]:
monzo = transactions.containing_tags('Monzo')
monzo.size()


2457

In [10]:
tags_involved = monzo.all_tags()
tags_involved

['Monzo',
 'All',
 'Spending',
 'MoneyOut',
 'Tap',
 'Afternoon',
 'Food',
 'Night',
 'Transfers',
 'Eating out',
 'MoneyIn',
 'My transfers',
 'Living costs',
 'Super Market',
 'Morning',
 'Commute',
 'Entertainment',
 'Drinks',
 'TFL',
 'Friends transfers',
 'BorisBike',
 'Online payments',
 'LimeBike',
 'Club',
 'Online orders',
 'Other bills',
 'Too good to go',
 'Rent',
 'Accomodation',
 'ITV income',
 'Income',
 'Alpha Bank',
 'GiffGaff',
 'Food delivery',
 'Flight tickets',
 'Train tickets',
 'Boat',
 'Uber Taxi',
 'Clothing',
 'Spotify']

In [4]:
import pandas as pd
old_calcs = pd.read_csv("/Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/v2_dataset_test/data/current/calc_monitoring.csv", index_col=None)
new_calcs = pd.read_csv("/Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/v2_dataset_test_monzo_api/data/current/calc_monitoring.csv", index_col=None)
old_calcs.shape, new_calcs.shape

((7643, 321), (7949, 321))

In [5]:
common_ids = set(old_calcs['id']).intersection(new_calcs['id'])
len(common_ids)

6315

In [6]:
common_cols = list(set(old_calcs.columns).intersection(set(new_calcs.columns)))
common_cols

['lower(description) contains cash',
 'lower(description) contains eating out',
 'lower(description) contains merieme katsani',
 'abs(amount) greater 34',
 'description contains EDF ENERGY-ECOM PLYMOUTH VIS',
 'split_comma(tags) in_csv Revolut,Monzo,HSBC',
 'description contains bank:HSBC, Amanmeet Singh Bha rent BP',
 'lower(description) contains drumshed',
 'lower(description) contains printworks',
 'lower(description) contains thames water',
 'lower(description) contains nando',
 'lower(description) contains mcdonald',
 'lower(description) contains blue star ferries',
 'str(datetime) greater 2023-04-06',
 'description contains bank:HSBCSVR',
 'lower(description) contains marsel katsani',
 'description contains bank:HSBC,',
 'lower(description) contains booking',
 'lower(description) contains tesco',
 'tags_added_1.8',
 'description contains Revolut**4562*',
 'lower(description) contains pophams',
 'description contains Dr Sina Motesharei rent BP',
 'hour(datetime) less 20',
 'lower(

In [7]:
fields = ['amount', 'datetime', 'description', 'amount_cur', 'currency', 'tags']

cols_of_interest = common_cols.copy()

for col in common_cols:
    for field in fields:
        if col == field:
            print(f"REMOVED: column '{col}' is field")
            cols_of_interest.remove(col)
            break
        elif col.startswith(field+'.'):
            print(f"REMOVED: column '{col}' is transformation")
            cols_of_interest.remove(col)
            break
        elif col.startswith('tags_list'):
            print(f"REMOVED: column '{col}' is tags_list")
            cols_of_interest.remove(col)
            break
        # else:
        #    print(f"ADDED: column {col} is VALID")
cols_of_interest

REMOVED: column 'amount' is field
REMOVED: column 'datetime.hour' is transformation
REMOVED: column 'tags' is field
REMOVED: column 'tags_list_3.8' is tags_list
REMOVED: column 'currency' is field
REMOVED: column 'amount_cur' is field
REMOVED: column 'amount.none' is transformation
REMOVED: column 'tags_list_2.8' is tags_list
REMOVED: column 'currency.none' is transformation
REMOVED: column 'description' is field
REMOVED: column 'description.upper' is transformation
REMOVED: column 'datetime.str' is transformation
REMOVED: column 'description.lower' is transformation
REMOVED: column 'amount.abs' is transformation
REMOVED: column 'tags_list_0.8' is tags_list
REMOVED: column 'tags_list_1.8' is tags_list
REMOVED: column 'datetime' is field
REMOVED: column 'description.none' is transformation


['lower(description) contains cash',
 'lower(description) contains eating out',
 'lower(description) contains merieme katsani',
 'abs(amount) greater 34',
 'description contains EDF ENERGY-ECOM PLYMOUTH VIS',
 'split_comma(tags) in_csv Revolut,Monzo,HSBC',
 'description contains bank:HSBC, Amanmeet Singh Bha rent BP',
 'lower(description) contains drumshed',
 'lower(description) contains printworks',
 'lower(description) contains thames water',
 'lower(description) contains nando',
 'lower(description) contains mcdonald',
 'lower(description) contains blue star ferries',
 'str(datetime) greater 2023-04-06',
 'description contains bank:HSBCSVR',
 'lower(description) contains marsel katsani',
 'description contains bank:HSBC,',
 'lower(description) contains booking',
 'lower(description) contains tesco',
 'tags_added_1.8',
 'description contains Revolut**4562*',
 'lower(description) contains pophams',
 'description contains Dr Sina Motesharei rent BP',
 'hour(datetime) less 20',
 'lower(

In [8]:
old = old_calcs[old_calcs['id'].isin(common_ids)].sort_values('id')[cols_of_interest].reset_index(drop=True)
new = new_calcs[new_calcs['id'].isin(common_ids)].sort_values('id')[cols_of_interest].reset_index(drop=True)
old.shape, new.shape

((6315, 231), (6315, 231))

In [9]:
# old['tags'] = old['tags'].apply(lambda s:set(s.split(',')))
# new['tags'] = new['tags'].apply(lambda s:set(s.split(',')))
# (old['tags'] == new['tags']).sum()

In [10]:
comps = old.compare(new)
# comps.columns = [c[0]+' ('+('old' if c[1]=='self' else 'new')+')' for c in comps.columns]
# cols = list({c for c in comps.columns})
cols = list({c[0] for c in comps.columns})

comps['id'] = old['id']
comps

lower(description) contains cash        \
                                 self other   
0                                 NaN   NaN   
1                                 NaN   NaN   
2                                 NaN   NaN   
3                                 NaN   NaN   
4                                 NaN   NaN   
...                               ...   ...   
6310                              NaN   NaN   
6311                              NaN   NaN   
6312                              NaN   NaN   
6313                              NaN   NaN   
6314                              NaN   NaN   

     lower(description) contains eating out       tags_added_1.8        \
                                       self other           self other   
0                                       NaN   NaN            NaN   NaN   
1                                       NaN   NaN            NaN   NaN   
2                                       NaN   NaN            NaN   NaN   
3                                       NaN   NaN            NaN   NaN   
4                                       NaN   NaN            NaN   NaN   
...                                     ...   ...            ...   ...   
6310                                    NaN   NaN            NaN   NaN   
6311                                    NaN   NaN            NaN   NaN   
6312                                    NaN   NaN            NaN   NaN   
6313                                    NaN   NaN            NaN   NaN   
6314                                    NaN   NaN            NaN   NaN   

     tags_added_3.8       tags_added_0.8        ...  \
               self other           self other  ...   
0               NaN   NaN            NaN   NaN  ...   
1               NaN   NaN            NaN   NaN  ...   
2               NaN   NaN            NaN   NaN  ...   
3               NaN   NaN            NaN   NaN  ...   
4               NaN   NaN            NaN   NaN  ...   
...             ...   ...            ...   ...  ...   
6310            NaN   NaN            NaN   NaN  ...   
6311            NaN   NaN            NaN   NaN  ...   
6312            NaN   NaN            NaN   NaN  ...   
6313            NaN   NaN            NaN   NaN  ...   
6314            NaN   NaN            NaN   NaN  ...   

     lower(description) contains category: eating out  \
                                                other   
0                                                 NaN   
1                                                 NaN   
2                                                 NaN   
3                                                 NaN   
4                                                 NaN   
...                                               ...   
6310                                              NaN   
6311                                              NaN   
6312                                              NaN   
6313                                              NaN   
6314                                              NaN   

     tags contains Food delivery        \
                            self other   
0                            NaN   NaN   
1                            NaN   NaN   
2                            NaN   NaN   
3                            NaN   NaN   
4                            NaN   NaN   
...                          ...   ...   
6310                         NaN   NaN   
6311                         NaN   NaN   
6312                         NaN   NaN   
6313                         NaN   NaN   
6314                         NaN   NaN   

                                               old_tags        \
                                                   self other   
0     HSBC Savings,MoneyIn,Night,My transfers,Spendi...   NaN   
1               HSBC Savings,MoneyIn,Night,Spending,All   NaN   
2               HSBC Savings,MoneyIn,Night,Spending,All   NaN   
3               HSBC Savings,MoneyIn,Night,Spending,All   NaN   
4               HSBC Savings,MoneyI

In [11]:
# cols = comps.columns
# cols = list({c[0] for c in comps.columns})
cols

['split_comma(tags) in_csv TFL,BorisBike,LimeBike,Uber Taxi',
 'lower(description) contains cash',
 'lower(description) contains eating out',
 'tags_added_2.8',
 'split_comma(tags) in_csv Eating out',
 'lower(description) contains category: entertainment',
 'lower(description) contains eat',
 'lower(description) contains transfer',
 'lower(description) contains lidl',
 'lower(description) contains hotel',
 'lower(description) contains ally',
 'tags_added_1.8',
 'split_comma(tags) in_csv Super Market',
 'tags not_contains Transfer',
 'lower(description) contains uber',
 'lower(description) contains restaurant',
 'tags_added_3.8',
 'lower(description) contains kfc',
 'lower(description) contains category: eating out',
 'lower(description) contains arms',
 'tags contains Food delivery',
 'lower(description) contains supermarket',
 'old_tags',
 'split_comma(tags) in_csv My transfers,Alpha Bank,Friends transfers,Currency exchange,Home Bills,Rent',
 'lower(description) not_contains eat',
 't

In [12]:
def get_col(col):
    res = comps[~comps[col].isna().any(axis=1)][['id', col]]
    res.columns = ['id', 'old', 'new']
    res['column'] = col
    res['count'] = len(res)
    return res

get_col(cols[1])

,id,old,new,column,count
997,MZNd20200203t133216an40000itx_00009rf6iRCr2zJC...,False,True,lower(description) contains cash,17
999,MZNd20200203t135813an10000itx_00009rf91yUbiIOO...,False,True,lower(description) contains cash,17
1006,MZNd20200206t195159an0itx_00009rls93fUWyi4uHLjKE,False,True,lower(description) contains cash,17
1007,MZNd20200206t195213an4000itx_00009rlsAKLM1CAwT...,False,True,lower(description) contains cash,17
1077,MZNd20200301t011821an40000itx_00009sY15bSTBTrW...,False,True,lower(description) contains cash,17
1082,MZNd20200302t200201an0itx_00009sbhsyDM925V2rNQrh,False,True,lower(description) contains cash,17
1083,MZNd20200302t200239an10000itx_00009sbhwQv1sYDE...,False,True,lower(description) contains cash,17
1084,MZNd20200302t200328an20000itx_00009sbi0yOVMNPx...,False,True,lower(description) contains cash,17
1104,MZNd20200307t220333an2000itx_00009smFIiEvOu98a...,False,True,lower(description) contains cash,17
1138,MZNd20200321t215135an3000itx_00009tFFQSh8iNQam...,False,True,lower(description) contains cash,17


In [13]:
all_diffs = pd.concat([get_col(col) for col in set(cols).difference(['id'])]).sort_values('count', ascending=False)
all_diffs

,id,old,new,column,count
1846,MZNd20241124t155659an2136itx_0000AoN9t3wJQT7nc...,False,True,lower(description) contains eat,877
2049,MZNd20250119t160333an1099itx_0000AqFFFNZQ9pxxQ...,False,True,lower(description) contains eat,877
2041,MZNd20250116t193104ap2000itx_0000Aq9KE6HmGm0Tk...,False,True,lower(description) contains eat,877
2042,MZNd20250116t193111an2026itx_0000Aq9KEiUAFADGE...,False,True,lower(description) contains eat,877
2043,MZNd20250118t143633an670itx_0000AqD2y7ZwlyvIgP...,False,True,lower(description) contains eat,877
...,...,...,...,...,...
2022,MZNd20250109t113848an260itx_0000Apu8TpklOJrBMP...,False,True,lower(description) contains uber,1
1523,MZNd20231216t222658an1200itx_0000AcsfwI1UBYA8Y...,"['All', 'Tap']",['All'],tags_added_3.8,1
2052,MZNd20250120t123831an270itx_0000AqH1Sljub8v9kZ...,False,True,lower(description) contains arms,1
1101,MZNd20200307t170556an4650itx_00009slojvB6hqg5R...,False,True,lower(description) contains lidl,1


In [14]:
comb_transactions = old_calcs[['id']+fields].add_prefix('old_').merge(new_calcs[['id']+fields].add_prefix('new_'), left_on='old_id', right_on='new_id')
comb_transactions['id'] = comb_transactions['old_id']
del comb_transactions['old_id'], comb_transactions['new_id']
comb_transactions

,old_amount,old_datetime,old_description,old_amount_cur,old_currency,old_tags,new_amount,new_datetime,new_description,new_amount_cur,new_currency,new_tags,id
0,8.333333,2019-02-27 15:03:15,"bank:Revolut, type: TRANSFER, product: Current...",10.00,EUR,"Afternoon,Friends transfers,MoneyIn,Revolut,My...",8.333333,2019-02-27 15:03:15,"bank:Revolut, type: TRANSFER, product: Current...",10.00,EUR,"Afternoon,Friends transfers,MoneyIn,Revolut,My...",RVLTd20190227t150315ap833i3634
1,-8.333333,2019-02-27 15:08:11,"bank:Revolut, type: TRANSFER, product: Current...",-10.00,EUR,"Afternoon,Friends transfers,MoneyOut,Revolut,M...",-8.333333,2019-02-27 15:08:11,"bank:Revolut, type: TRANSFER, product: Current...",-10.00,EUR,"Afternoon,Friends transfers,MoneyOut,Revolut,M...",RVLTd20190227t150811an833i3635
2,6.666667,2019-04-11 09:06:18,"bank:Revolut, type: TOPUP, product: Current, c...",8.00,EUR,"Alpha Bank,MoneyIn,Morning,Revolut,My transfer...",6.666667,2019-04-11 09:06:18,"bank:Revolut, type: TOPUP, product: Current, c...",8.00,EUR,"Alpha Bank,MoneyIn,Morning,Revolut,My transfer...",RVLTd20190411t090618ap666i3636
3,-4.850000,2019-04-13 18:06:40,"bank:Revolut, type: CARD_PAYMENT, product: Cur...",-5.82,EUR,"Afternoon,GiffGaff,MoneyOut,Revolut,Other bill...",-4.850000,2019-04-13 18:06:40,"bank:Revolut, type: CARD_PAYMENT, product: Cur...",-5.82,EUR,"Afternoon,GiffGaff,MoneyOut,Revolut,Other bill...",RVLTd20190413t180640an485i3637
4,-0.966667,2019-04-25 10:06:35,"bank:Revolut, type: TRANSFER, product: Current...",-1.16,EUR,"Alpha Bank,MoneyOut,Morning,Revolut,My transfe...",-0.966667,2019-04-25 10:06:35,"bank:Revolut, type: TRANSFER, product: Current...",-1.16,EUR,"Alpha Bank,MoneyOut,Morning,Revolut,My transfe...",RVLTd20190425t100635an96i3638
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6310,-25.140000,2025-02-03 20:21:36,"bank:Monzo, type: Card payment, name: Lidl, em...",-25.14,GBP,"MoneyOut,Monzo,Night,Super Market,Food,Living ...",-25.140000,2025-02-03 20:21:36,"bank:MonzoAPI, other_fields: {merchant_feedbac...",-25.14,GBP,"MoneyOut,Monzo,Night,Super Market,Food,Living ...",MZNd20250203t202136an2514itx_0000Aqkhz36zNJ4iA...
6311,500.000000,2025-02-05 14:39:09,"bank:TRD212, action: Deposit, notes: Transacti...",500.00,GBP,"Afternoon,MoneyIn,Trading212,My transfers,Spen...",500.000000,2025-02-05 14:39:09,"bank:TRD212, action: Deposit, notes: Transacti...",500.00,GBP,"Afternoon,MoneyIn,Trading212,My transfers,Spen...",TRD212d20250205t143909ap50000i5
6312,-2642.740000,2025-02-07 19:14:24,"bank:TRD212, action: Withdrawal, raw_id: 8ea50...",-2642.74,GBP,"Afternoon,MoneyOut,Trading212,Spending,All",-2642.740000,2025-02-07 19:14:24,"bank:TRD212, action: Withdrawal, raw_id: 8ea50...",-2642.74,GBP,"Afternoon,MoneyOut,Trading212,Spending,All",TRD212d20250207t191424an264274i6
6313,3642.740000,2025-02-11 16:35:10,"bank:TRD212, action: Deposit, raw_id: 76b170f5...",3642.74,GBP,"Afternoon,MoneyIn,Trading212,Spending,All",3642.740000,2025-02-11 16:35:10,"bank:TRD212, action: Deposit, raw_id: 76b170f5...",3642.74,GBP,"Afternoon,MoneyIn,Trading212,Spending,All",TRD212d20250211t163510ap364274i7


In [17]:
all_diffs[all_diffs['column']=='lower(description) contains eat'].merge(comb_transactions[['id', 'old_description', 'new_description']], on='id')

,id,old,new,column,count,old_description,new_description
0,MZNd20241124t155659an2136itx_0000AoN9t3wJQT7nc...,False,True,lower(description) contains eat,877,"bank:Monzo, type: Card payment, name: Willows,...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
1,MZNd20250119t160333an1099itx_0000AqFFFNZQ9pxxQ...,False,True,lower(description) contains eat,877,"bank:Monzo, type: Card payment, name: Barrys F...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
2,MZNd20250116t193104ap2000itx_0000Aq9KE6HmGm0Tk...,False,True,lower(description) contains eat,877,"bank:Monzo, type: Faster payment, name: KONTZE...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
3,MZNd20250116t193111an2026itx_0000Aq9KEiUAFADGE...,False,True,lower(description) contains eat,877,"bank:Monzo, type: Card payment, name: Lidl, em...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
4,MZNd20250118t143633an670itx_0000AqD2y7ZwlyvIgP...,False,True,lower(description) contains eat,877,"bank:Monzo, type: Card payment, name: Tesco, e...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
...,...,...,...,...,...,...,...
872,MZNd20230207t140457an1250itx_0000ASREpNmBzRNSu...,False,True,lower(description) contains eat,877,"bank:Monzo, type: Card payment, name: Transpor...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
873,MZNd20230205t162157an509itx_0000ASNI1fsq14roNz...,False,True,lower(description) contains eat,877,"bank:Monzo, type: Card payment, name: Ally S W...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
874,MZNd20230206t122729an0itx_0000ASP1c8x26w7GvCaREn,False,True,lower(description) contains eat,877,"bank:Monzo, type: Card payment, name: Google, ...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
875,MZNd20230206t122906ap8000itx_0000ASP1lAFodUaqY...,False,True,lower(description) contains eat,877,"bank:Monzo, type: Faster payment, name: Dimitr...","bank:MonzoAPI, other_fields: {merchant_feedbac..."
